In [ ]:
#ARRANGER FM

In [1]:
import os
import subprocess

import mss
import win32gui
import numpy as np
from PIL import Image, ImageDraw

from pathlib import Path

import vgamepad as vg
import time

In [2]:
# 1. Ścieżka do gry
game_path = r"E:\gry_laptop\Arranger"
#game_name = "Arranger"

#print(f"Uruchamiam grę: {game_name}")
#game = subprocess.Popen(os.path.join(game_path, game_name+".exe"), cwd=game_path)

In [3]:
class WindowCapture:
    def __init__(self, window_title):
        self.window_title = window_title
        self.hwnd = None

        self.left = 0
        self.top = 0
        self.width = 0
        self.height = 0
        self.update_window() #od razu formuje

    def update_window(self):
        self.hwnd = win32gui.FindWindow(None, self.window_title)
        if not self.hwnd:
            raise Exception("Window not found")

        l, t, r, b = win32gui.GetWindowRect(self.hwnd)

        self.left = l
        self.top = t
        self.width = r - l
        self.height = b - t

    def get_screenshot(self):
        with mss.mss() as sct:
            monitor = {
                "left": self.left+8,
                "top": self.top+31,
                "width": self.width-16,
                "height": self.height-39
            }

            img = sct.grab(monitor)
            #frame = np.array(img)[:,:,:4][:,:,:-1]  # BGRA
            frame = np.array(img)[:,:,:3][:,:,::-1]  # BGRA
            #mss.tools.to_png(img.rgb, img.size, output="window.png")
            return frame

In [11]:
class ArrangerGame():
    def __init__(self,game_path):
        game_name = "Arranger"
        print(f"Uruchamiam grę: {game_name}")
        g = subprocess.Popen(os.path.join(game_path, game_name+".exe"), cwd=game_path)

        self.gamepad = vg.VX360Gamepad()
        self.moveset = {"A":vg.XUSB_BUTTON.XUSB_GAMEPAD_A
                        ,"B":vg.XUSB_BUTTON.XUSB_GAMEPAD_B
                        ,"U":vg.XUSB_BUTTON.XUSB_GAMEPAD_DPAD_UP
                        ,"L":vg.XUSB_BUTTON.XUSB_GAMEPAD_DPAD_LEFT
                        ,"D":vg.XUSB_BUTTON.XUSB_GAMEPAD_DPAD_DOWN
                        ,"R":vg.XUSB_BUTTON.XUSB_GAMEPAD_DPAD_RIGHT
                        }
    
    def one_step(self,click,prep=None):
        if prep:
            time.sleep(prep) #czas na wyklikanie
        self.gamepad.press_button(button=self.moveset[click.upper()])
        self.gamepad.update()
        
        time.sleep(0.05)
        
        self.gamepad.release_button(button=self.moveset[click.upper()])
        self.gamepad.update()

        time.sleep(0.2)
        return
    
    def route(self,ll,timeout=5,w=0.02):
        print("!",ll)
        for m in list(ll):
            self.one_step(m,w)
        
        time.sleep(timeout)
        return

In [ ]:
def get_line(prefix="0_0",sciezka="solutions") -> str:
    katalog = Path(sciezka)

    if not katalog.is_dir():
        raise NotADirectoryError(f"Nieprawidłowy katalog: {sciezka}")

    pliki = sorted(
        plik
        for plik in katalog.iterdir()
        if plik.is_file() and plik.name.startswith(prefix)
    )

    if not pliki:
        raise FileNotFoundError(
            f"Nie znaleziono plików z prefiksem '{prefix}' w katalogu '{sciezka}'"
        )

    with pliki[0].open("r", encoding="utf-8") as plik:
        return plik.readline().rstrip("\r\n")

In [6]:
game = ArrangerGame(game_path)
game.one_step("A",30)
#w = WindowCapture("Arranger")
game.route("LLLDADAADDAADDAA",25)

Uruchamiam grę: Arranger
! LLLDADAADDAADDAA


In [10]:
time.sleep(6)
game.route("LLL",timeout=1.5)
#game.route("DLDRUULDLLDLUUUULDRDRDDDRURUUUULUUUULDDRDLLLUURDLUURDLDLLURUU",2) # my perfect
#game.route("DLUULDRDRRDRUUUULDRUURDRURUUUULUUUULDDDRUURDLUURDLDLLULUU",2) #PC BETTER

game.route("A"+get_line("0_0")[3:]+"A"*32+"R")
#print("ONE STEP AWAY")

! LLL
! ADLDLUURDRRDRUUUULDRUURDRRUUUURDRUUULDDDRUURDLUURDLDLLULUUUAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAR


In [12]:
time.sleep(6)
game.route("RRRR" + get_line("1_1")[:3]+"A"*32+get_line("1_1")[3:])
#print("ONE STEP AWAY")

! RRRRRUULDLLLURRDRULDLLLLLLUULULLLUULLLDAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAR


In [22]:
get_line()

'LLLDLDLUURDRRDRUUUULDRUURDRRUUUURDRUUULDDDRUURDLUURDLDLLULUUU'